# Social Intelligence with XPOZ + Claude

This cookbook shows how to use [XPOZ](https://xpoz.ai) — a social intelligence platform with 1.5B+ indexed posts across Twitter, Instagram, Reddit & TikTok — together with Claude for real-time social media analysis.

**What you'll learn:**
1. Setup: Install the XPOZ Python SDK and configure API keys
2. Brand Sentiment Analysis: Fetch social posts and classify sentiment with Claude
3. Competitive Intelligence: Compare two brands' social footprints
4. Influencer Discovery: Find the top voices on any topic
5. Real-time Event Monitoring: Social overlay for prediction markets

**Prerequisites:**
- XPOZ API key (free at [xpoz.ai/get-token](https://xpoz.ai/get-token) — 100K results/month)
- Anthropic API key ([console.anthropic.com](https://console.anthropic.com))

## 1. Setup

In [ ]:
# Install dependencies
!pip install xpoz anthropic

In [ ]:
import os

# Set your API keys (or use environment variables)
os.environ["XPOZ_API_KEY"] = "your-xpoz-api-key"  # Get at xpoz.ai/get-token
os.environ["ANTHROPIC_API_KEY"] = "your-anthropic-api-key"  # Get at console.anthropic.com

In [ ]:
from xpoz import XpozClient
import anthropic
import json
from datetime import datetime, timedelta

# Initialize clients
xpoz = XpozClient()
claude = anthropic.Anthropic()

# Helper: get date N days ago
def days_ago(n: int) -> str:
    return (datetime.now() - timedelta(days=n)).strftime("%Y-%m-%d")

print("Clients initialized successfully!")

## 2. Brand Sentiment Analysis

Fetch recent social media posts about a brand and use Claude to classify sentiment, extract themes, and identify risks.

In [ ]:
# Step 1: Fetch social data from XPOZ
BRAND = "NVIDIA"

# Search Twitter
twitter_results = xpoz.twitter.search_posts(
    BRAND,
    start_date=days_ago(7),
    limit=50,
    fields=["text", "author_username", "like_count", "retweet_count", "created_at_date"]
)

# Search Reddit
reddit_results = xpoz.reddit.search_posts(
    BRAND,
    start_date=days_ago(7),
    limit=30,
    fields=["title", "selftext", "author_username", "score", "subreddit_name", "created_at_date"]
)

# Count total volume
tweet_count = xpoz.twitter.count_posts(BRAND, start_date=days_ago(7))

print(f"Found {len(twitter_results.data)} tweets, {len(reddit_results.data)} Reddit posts")
print(f"Total tweet volume (7 days): {tweet_count:,}")

In [ ]:
# Step 2: Analyze with Claude
tweets_text = "\n".join([
    f"@{p.author_username}: {p.text} [likes: {p.like_count}, RTs: {p.retweet_count}]"
    for p in twitter_results.data
])

reddit_text = "\n".join([
    f"r/{p.subreddit_name} — {p.title}: {(p.selftext or '')[:200]} [score: {p.score}]"
    for p in reddit_results.data
])

response = claude.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=2048,
    messages=[{
        "role": "user",
        "content": f"""Analyze the social media sentiment for {BRAND} based on these posts from the last 7 days.
Total tweet volume: {tweet_count:,}

Twitter posts:
{tweets_text}

Reddit posts:
{reddit_text}

Provide:
1. Overall Sentiment (Bullish/Bearish/Neutral with confidence %)
2. Key Themes (top 5 with sentiment for each)
3. Risk Signals (emerging issues or negative trends)
4. Opportunities (positive trends to capitalize on)
5. Most Impactful Posts (3 highest-engagement posts)"""
    }]
)

print(response.content[0].text)

## 3. Competitive Intelligence

Compare two brands' social media presence — volume, sentiment, themes, and share of voice.

In [ ]:
# Compare two brands
BRAND_A = "NVIDIA"
BRAND_B = "AMD"

# Fetch data for both brands
def fetch_brand_data(brand: str):
    tweets = xpoz.twitter.search_posts(
        brand, start_date=days_ago(7), limit=50,
        fields=["text", "author_username", "like_count", "retweet_count"]
    )
    reddit = xpoz.reddit.search_posts(
        brand, start_date=days_ago(7), limit=30,
        fields=["title", "selftext", "score", "subreddit_name"]
    )
    count = xpoz.twitter.count_posts(brand, start_date=days_ago(7))
    return {
        "brand": brand,
        "tweet_count": count,
        "tweets": [p.text for p in tweets.data],
        "reddit_posts": [f"{p.title}: {(p.selftext or '')[:100]}" for p in reddit.data],
        "avg_engagement": sum(p.like_count or 0 for p in tweets.data) / max(len(tweets.data), 1)
    }

data_a = fetch_brand_data(BRAND_A)
data_b = fetch_brand_data(BRAND_B)

print(f"{BRAND_A}: {data_a['tweet_count']:,} tweets, avg {data_a['avg_engagement']:.0f} likes")
print(f"{BRAND_B}: {data_b['tweet_count']:,} tweets, avg {data_b['avg_engagement']:.0f} likes")
print(f"Share of Voice: {BRAND_A} {data_a['tweet_count']/(data_a['tweet_count']+data_b['tweet_count'])*100:.0f}% / {BRAND_B} {data_b['tweet_count']/(data_a['tweet_count']+data_b['tweet_count'])*100:.0f}%")

In [ ]:
# Competitive analysis with Claude
response = claude.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=2048,
    messages=[{
        "role": "user",
        "content": f"""Compare these two brands' social media presence over the last 7 days:

{BRAND_A}:
- Tweet volume: {data_a['tweet_count']:,}
- Sample tweets: {json.dumps(data_a['tweets'][:10])}
- Reddit: {json.dumps(data_a['reddit_posts'][:5])}

{BRAND_B}:
- Tweet volume: {data_b['tweet_count']:,}
- Sample tweets: {json.dumps(data_b['tweets'][:10])}
- Reddit: {json.dumps(data_b['reddit_posts'][:5])}

Provide a competitive intelligence brief:
1. Share of Voice comparison
2. Sentiment comparison (which brand has more positive/negative coverage?)
3. Key differentiators in public perception
4. Emerging narratives that could shift the competitive landscape
5. Strategic recommendations for each brand"""
    }]
)

print(response.content[0].text)

## 4. Influencer Discovery

Find the most influential voices discussing a topic across Twitter.

In [ ]:
TOPIC = '"artificial intelligence" AND ethics'

# Find users who post about this topic
users = xpoz.twitter.get_users_by_keywords(
    TOPIC,
    start_date=days_ago(30),
    limit=20,
    fields=["username", "name", "followers_count", "description", "verified",
            "relevant_tweets_count", "relevant_tweets_impressions_sum"]
)

print(f"Found {len(users.data)} influencers on '{TOPIC}':\n")
for u in sorted(users.data, key=lambda x: x.followers_count or 0, reverse=True)[:10]:
    verified = " \u2713" if u.verified else ""
    print(f"@{u.username}{verified} — {(u.followers_count or 0):,} followers")
    print(f"  {(u.description or '')[:80]}")
    print(f"  {u.relevant_tweets_count or 0} relevant posts, {(u.relevant_tweets_impressions_sum or 0):,} impressions")
    print()

In [ ]:
# Deep-dive: get recent posts from the top influencer
if users.data:
    top_user = sorted(users.data, key=lambda x: x.followers_count or 0, reverse=True)[0]
    
    posts = xpoz.twitter.get_posts_by_author(
        top_user.username,
        start_date=days_ago(14),
        limit=10,
        fields=["text", "like_count", "retweet_count", "impression_count", "created_at_date"]
    )
    
    print(f"Recent posts from @{top_user.username} ({(top_user.followers_count or 0):,} followers):\n")
    for p in posts.data:
        print(f"{p.created_at_date} — {(p.text or '')[:120]}")
        print(f"  Likes: {p.like_count:,} | RTs: {p.retweet_count:,} | Impressions: {(p.impression_count or 0):,}")
        print()

## 5. Real-time Event Monitoring

Monitor social sentiment around real-world events — useful for prediction markets, trading signals, and crisis detection.

In [ ]:
# Example: Monitor social signals for a prediction market event
EVENT = "Bitcoin price"
QUERY_YES = '"bitcoin" AND ("bullish" OR "moon" OR "buy" OR "breakout" OR "rally" OR "ATH")'
QUERY_NO = '"bitcoin" AND ("bearish" OR "crash" OR "sell" OR "dump" OR "bubble" OR "overvalued")'

# Count bullish vs bearish sentiment
bullish_count = xpoz.twitter.count_posts(QUERY_YES, start_date=days_ago(3))
bearish_count = xpoz.twitter.count_posts(QUERY_NO, start_date=days_ago(3))
total = bullish_count + bearish_count

print(f"Social Sentiment for '{EVENT}' (last 3 days):")
print(f"  Bullish: {bullish_count:,} posts ({bullish_count/max(total,1)*100:.0f}%)")
print(f"  Bearish: {bearish_count:,} posts ({bearish_count/max(total,1)*100:.0f}%)")
print(f"  Sentiment Ratio: {bullish_count/max(bearish_count,1):.1f}x bullish")

In [ ]:
# Get the most impactful posts from both sides
bullish_posts = xpoz.twitter.search_posts(
    QUERY_YES, start_date=days_ago(3), limit=20,
    fields=["text", "author_username", "like_count", "retweet_count", "impression_count"]
)

bearish_posts = xpoz.twitter.search_posts(
    QUERY_NO, start_date=days_ago(3), limit=20,
    fields=["text", "author_username", "like_count", "retweet_count", "impression_count"]
)

# Use Claude to synthesize
response = claude.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=2048,
    messages=[{
        "role": "user",
        "content": f"""Analyze the social sentiment around "{EVENT}" to produce a prediction signal.

Overall volume (3 days): {bullish_count:,} bullish vs {bearish_count:,} bearish posts.

Bullish posts (sample):
{chr(10).join(f'@{p.author_username}: {p.text} [likes:{p.like_count}]' for p in bullish_posts.data[:10])}

Bearish posts (sample):
{chr(10).join(f'@{p.author_username}: {p.text} [likes:{p.like_count}]' for p in bearish_posts.data[:10])}

Provide:
1. Sentiment Signal: BULLISH / BEARISH / NEUTRAL (with confidence 1-10)
2. Key Arguments: Top 3 arguments from each side
3. Influential Voices: Who is driving the narrative on each side?
4. Catalysts: Recent events or news driving sentiment
5. Risk Assessment: What could flip the sentiment?"""
    }]
)

print(response.content[0].text)

## Using XPOZ as an MCP Tool with Claude

XPOZ can also be used as an MCP (Model Context Protocol) server, giving Claude direct access to search social media.

### Claude Desktop Setup

Add this to your Claude Desktop MCP config (`~/Library/Application Support/Claude/claude_desktop_config.json`):

```json
{
  "mcpServers": {
    "xpoz": {
      "url": "https://mcp.xpoz.ai/mcp",
      "headers": {
        "Authorization": "Bearer YOUR_XPOZ_API_KEY"
      }
    }
  }
}
```

### Claude Code Setup

```bash
claude mcp add xpoz https://mcp.xpoz.ai/mcp \
  --header "Authorization: Bearer YOUR_XPOZ_API_KEY"
```

Once configured, Claude can directly search Twitter, Instagram, Reddit, and TikTok using XPOZ's 29+ tools.

In [ ]:
# Clean up
xpoz.close()
print("Done! XPOZ client closed.")